# 21. Colocalization Analysis: AD GWAS × Colon eQTL (Full Summary Statistics)## PurposeTest whether the AD GWAS signal and colon eQTL signal at each locus share the **same causal variant** (colocalization).**Previous analysis** used only significant eQTL variants from the GTEx API (~40-200 SNPs per locus).  **This analysis** uses the **full eQTL summary statistics** from the eQTL Catalogue (~5,000-15,000 SNPs per locus), which is the methodologically correct approach.## Method: coloc-ABF (Approximate Bayes Factor)- For each locus, extract ALL SNPs from both AD GWAS and colon eQTL- Compute log Approximate Bayes Factors (Wakefield 2009)- Run Bayesian colocalization test (Giambartolomei et al. 2014)- Posterior probabilities for 5 hypotheses:  - **H0**: No association with either trait  - **H1**: GWAS association only  - **H2**: eQTL association only    - **H3**: Both associated, **different** causal variants  - **H4**: Both associated, **shared** causal variant ← we want this## Data Sources- **AD GWAS**: Bellenguez et al. 2022 (Nat Genet), N=788,989, GRCh38- **Colon eQTL**: GTEx V8 via eQTL Catalogue (EMBL-EBI), full cis-eQTL associations  - Colon Transverse: ~406 samples  - Colon Sigmoid: ~373 samples## Genes TestedAPP pathway: APP, ADAM10, PICALM, CD2AP, SORL1, PSEN1, CR1  Myeloid (control): TREM2, SPI1, BIN1

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom scipy import statsfrom coloc import coloc as run_colocimport gzip, warnings, mathwarnings.filterwarnings('ignore')%matplotlib inlineplt.rcParams.update({'font.family': 'Arial', 'font.size': 11, 'pdf.fonttype': 42})BASE = '../analysis/26_gsmap'FIG = f'{BASE}/figures'print('Ready')

## Step 1: Load AD GWAS Summary Statistics

In [ ]:
# Load AD GWAS (Bellenguez 2022, GRCh38)gwas = pd.read_csv(f'{BASE}/data/gwas/Bellenguez2022_AD_withN.tsv.gz', sep='\t',    usecols=['variant_id','p_value','chromosome','base_pair_location','beta','standard_error','N',             'effect_allele','other_allele'])gwas = gwas.rename(columns={'variant_id':'rsid', 'base_pair_location':'pos', 'chromosome':'chr',                             'standard_error':'se'})gwas['chr'] = gwas['chr'].astype(str)gwas = gwas.dropna(subset=['beta','se'])gwas = gwas[gwas['se'] > 0]print(f"AD GWAS: {len(gwas):,} variants")gwas.head(3)

## Step 2: Load GTEx V8 Colon eQTL (Full Summary Statistics)

In [ ]:
# Load full eQTL data for a specific gene region# eQTL Catalogue format: variant, r2, pvalue, molecular_trait_id, ..., beta, se, ..., rsiddef load_eqtl_region(tissue_file, gene_id, chrom, start, end):    """Load all eQTL associations for a gene in a genomic region."""    results = []    with gzip.open(tissue_file, 'rt') as f:        header = f.readline().strip().split('\t')        for line in f:            fields = line.strip().split('\t')            row = dict(zip(header, fields))                        # Filter to gene and region            if row.get('gene_id','').split('.')[0] != gene_id.split('.')[0]:                continue                        try:                pos = int(row['position'])                if row['chromosome'] != str(chrom) or pos < start or pos > end:                    continue            except:                continue                        try:                results.append({                    'rsid': row.get('rsid', ''),                    'pos': pos,                    'chr': row['chromosome'],                    'beta_eqtl': float(row['beta']),                    'se_eqtl': float(row['se']),                    'p_eqtl': float(row['pvalue']),                    'maf': float(row.get('maf', 0)),                    'gene_id': row.get('gene_id', ''),                })            except:                continue        return pd.DataFrame(results)# Test with ADAM10 locusprint("Loading ADAM10 eQTL from Colon_Transverse (full file scan - may take a few minutes)...")eqtl_test = load_eqtl_region(    f'{BASE}/data/gtex_eqtl/Colon_Transverse.tsv.gz',    'ENSG00000137845', '15', 58200000, 59400000)print(f"ADAM10 Colon_Transverse: {len(eqtl_test)} SNPs (ALL tested, not just significant)")print(f"Min p-value: {eqtl_test['p_eqtl'].min():.2e}")eqtl_test.head()

## Step 3: Define Loci and Run Colocalization

In [ ]:
# Gene regions (GRCh38, ±500kb)loci = {    'PICALM':  {'gene_id': 'ENSG00000073921', 'chr': '11', 'start': 85457684, 'end': 86569882, 'ad_snp': 'rs10792832'},    'CD2AP':   {'gene_id': 'ENSG00000198087', 'chr': '6',  'start': 70650000, 'end': 72000000, 'ad_snp': 'rs10948363'},    'APP':     {'gene_id': 'ENSG00000142192', 'chr': '21', 'start': 25380550, 'end': 26671128, 'ad_snp': 'rs2154481'},    'SORL1':   {'gene_id': 'ENSG00000137642', 'chr': '11', 'start': 121324073, 'end': 121878524, 'ad_snp': 'rs11218343'},    'ADAM10':  {'gene_id': 'ENSG00000137845', 'chr': '15', 'start': 58200000, 'end': 59400000, 'ad_snp': 'rs593742'},    'PSEN1':   {'gene_id': 'ENSG00000080815', 'chr': '14', 'start': 73136417, 'end': 73723829, 'ad_snp': 'rs165932'},    'CR1':     {'gene_id': 'ENSG00000203710', 'chr': '1',  'start': 206496083, 'end': 208782987, 'ad_snp': 'rs6656401'},    # Myeloid controls    'TREM2':   {'gene_id': 'ENSG00000095970', 'chr': '6',  'start': 40838000, 'end': 41545000, 'ad_snp': 'rs143332484'},    'SPI1':    {'gene_id': 'ENSG00000066336', 'chr': '11', 'start': 47080000, 'end': 47680000, 'ad_snp': 'rs10437655'},    'BIN1':    {'gene_id': 'ENSG00000136717', 'chr': '2',  'start': 127035000, 'end': 127635000, 'ad_snp': 'rs6733839'},}tissues = {    'Colon_Transverse': f'{BASE}/data/gtex_eqtl/Colon_Transverse.tsv.gz',    'Colon_Sigmoid':    f'{BASE}/data/gtex_eqtl/Colon_Sigmoid.tsv.gz',}print(f"Loci to test: {len(loci)}")print(f"Tissues: {list(tissues.keys())}")

In [ ]:
# Coloc ABF functiondef compute_lnabf(beta, se, W=0.04):    """Log Approximate Bayes Factor (Wakefield 2009)"""    z = beta / se    r = W / (W + se**2)    lnabf = 0.5 * (np.log(1 - r) + r * z**2)    return lnabfdef run_coloc_analysis(gwas_df, eqtl_df, gene_name, tissue_name):    """Run coloc for one gene-tissue pair using full summary stats."""    # Merge on rsid    merged = gwas_df.merge(eqtl_df, on='rsid', how='inner', suffixes=('_gwas','_eqtl'))        if len(merged) < 50:        # Try position-based merge        merged = gwas_df.merge(eqtl_df, left_on='pos', right_on='pos', how='inner', suffixes=('_gwas','_eqtl'))        if len(merged) < 50:        return None        # Remove duplicates and invalid entries    merged = merged.drop_duplicates(subset=['pos'])    merged = merged[(merged['se'] > 0) & (merged['se_eqtl'] > 0)]    merged = merged.dropna(subset=['beta','se','beta_eqtl','se_eqtl'])        if len(merged) < 50:        return None        # Compute log ABF    merged['lnabf_gwas'] = compute_lnabf(merged['beta'].values, merged['se'].values, W=0.04)    merged['lnabf_eqtl'] = compute_lnabf(merged['beta_eqtl'].values, merged['se_eqtl'].values, W=0.04)        # Run coloc    pp = list(run_coloc(        merged['lnabf_gwas'].tolist(),        merged['lnabf_eqtl'].tolist(),        prior1=1e-4, prior2=1e-4, prior12=1e-5    ))        # Find top SNPs    gwas_top_idx = merged['lnabf_gwas'].idxmax()    eqtl_top_idx = merged['lnabf_eqtl'].idxmax()        return {        'gene': gene_name, 'tissue': tissue_name,        'n_snps': len(merged),        'PP_H0': pp[0], 'PP_H1': pp[1], 'PP_H2': pp[2], 'PP_H3': pp[3], 'PP_H4': pp[4],        'gwas_top_snp': merged.loc[gwas_top_idx, 'rsid_gwas'] if 'rsid_gwas' in merged else '',        'gwas_top_p': merged.loc[gwas_top_idx, 'p_value'] if 'p_value' in merged else None,        'eqtl_top_snp': merged.loc[eqtl_top_idx, 'rsid_eqtl'] if 'rsid_eqtl' in merged else '',        'eqtl_top_p': merged.loc[eqtl_top_idx, 'p_eqtl'],        'merged_data': merged,  # Keep for visualization    }

## Step 4: Run Coloc for All Genes × Tissues

In [ ]:
# Main analysis loopall_results = []for gene_name, locus in loci.items():    print(f"\n{'='*60}")    print(f"Analyzing {gene_name} (chr{locus['chr']}:{locus['start']}-{locus['end']})")        # Get GWAS data for region    gwas_region = gwas[(gwas['chr']==locus['chr']) &                        (gwas['pos']>=locus['start']) &                        (gwas['pos']<=locus['end'])].copy()    print(f"  GWAS SNPs: {len(gwas_region)}")        if len(gwas_region) < 100:        print(f"  Too few GWAS SNPs, skipping")        continue        gwas_top = gwas_region.loc[gwas_region['p_value'].idxmin()]    print(f"  GWAS top: {gwas_top['rsid']} p={gwas_top['p_value']:.2e}")        for tissue_name, tissue_file in tissues.items():        print(f"  Loading {tissue_name} eQTL...")        eqtl = load_eqtl_region(tissue_file, locus['gene_id'], locus['chr'], locus['start'], locus['end'])                if len(eqtl) < 50:            print(f"    {tissue_name}: {len(eqtl)} eQTL SNPs (too few)")            continue                print(f"    {tissue_name}: {len(eqtl)} eQTL SNPs")        eqtl_top = eqtl.loc[eqtl['p_eqtl'].idxmin()]        print(f"    eQTL top: {eqtl_top['rsid']} p={eqtl_top['p_eqtl']:.2e}")                # Run coloc        result = run_coloc_analysis(gwas_region, eqtl, gene_name, tissue_name)                if result:            all_results.append(result)            print(f"    COLOC ({result['n_snps']} matched SNPs):")            print(f"      PP.H0={result['PP_H0']:.4f}  H1={result['PP_H1']:.4f}  "                  f"H2={result['PP_H2']:.4f}  H3={result['PP_H3']:.4f}  H4={result['PP_H4']:.4f}")            if result['PP_H4'] > 0.5:                print(f"      *** COLOCALIZED ***")        else:            print(f"    Could not run coloc (insufficient matched SNPs)")print(f"\nTotal results: {len(all_results)}")

## Step 5: Summary Table

In [ ]:
# Summarysummary = pd.DataFrame([{k:v for k,v in r.items() if k != 'merged_data'} for r in all_results])print("=" * 100)print("COLOCALIZATION SUMMARY (Full eQTL Summary Statistics)")print("=" * 100)print(f"{'Gene':10s} {'Tissue':20s} {'N_SNPs':>7s} {'PP.H3':>7s} {'PP.H4':>7s} {'Interpretation'}")print("-" * 100)for _, r in summary.iterrows():    if r['PP_H4'] > 0.8:        interp = "STRONG coloc"    elif r['PP_H4'] > 0.5:        interp = "Moderate coloc"    elif r['PP_H3'] > 0.5:        interp = "Different causal variants"    elif r['PP_H1'] > 0.5:        interp = "GWAS only"    elif r['PP_H2'] > 0.5:        interp = "eQTL only"    else:        interp = "Inconclusive"    print(f"{r['gene']:10s} {r['tissue']:20s} {r['n_snps']:7d} {r['PP_H3']:7.4f} {r['PP_H4']:7.4f} {interp}")summary.to_csv(f'{BASE}/results/coloc_full_eqtl_results.csv', index=False)print(f"\nSaved to results/coloc_full_eqtl_results.csv")

## Step 6: Visualization — LocusZoom-style Plots

In [ ]:
# LocusZoom-style plots for top colocalizing locicoloc_results = [r for r in all_results if r['PP_H4'] > 0.3]  # Show all interesting onesif not coloc_results:    print("No colocalizing loci found — check results above")else:    n_plots = len(coloc_results)    fig, axes = plt.subplots(n_plots, 2, figsize=(16, 4*n_plots), squeeze=False)        for i, result in enumerate(coloc_results):        merged = result['merged_data']        gene = result['gene']        tissue = result['tissue']        pp4 = result['PP_H4']                # Left: GWAS signal        ax = axes[i, 0]        logp_gwas = -np.log10(merged['p_value'].clip(1e-50))        ax.scatter(merged['pos']/1e6, logp_gwas, s=3, alpha=0.6, c='steelblue', rasterized=True)        ax.set_ylabel('-log10(p) GWAS', fontsize=10)        ax.set_title(f'{gene} — AD GWAS', fontsize=11, fontweight='bold')        if i == 0:            ax.text(-0.1, 1.1, 'GWAS', transform=ax.transAxes, fontsize=14, fontweight='bold', color='steelblue')                # Highlight top SNP        top_idx = logp_gwas.idxmax()        ax.scatter(merged.loc[top_idx, 'pos']/1e6, logp_gwas[top_idx], s=80, c='red',                   zorder=5, edgecolors='black', linewidth=0.5)                # Right: eQTL signal          ax = axes[i, 1]        logp_eqtl = -np.log10(merged['p_eqtl'].clip(1e-50))        ax.scatter(merged['pos']/1e6, logp_eqtl, s=3, alpha=0.6, c='#E74C3C', rasterized=True)        ax.set_ylabel(f'-log10(p) {tissue}', fontsize=10)        ax.set_title(f'{gene} — {tissue} eQTL (PP.H4={pp4:.3f})', fontsize=11, fontweight='bold')        if i == 0:            ax.text(-0.1, 1.1, 'eQTL', transform=ax.transAxes, fontsize=14, fontweight='bold', color='#E74C3C')                top_idx_e = logp_eqtl.idxmax()        ax.scatter(merged.loc[top_idx_e, 'pos']/1e6, logp_eqtl[top_idx_e], s=80, c='red',                  zorder=5, edgecolors='black', linewidth=0.5)                # Add x label to bottom row        if i == n_plots - 1:            axes[i, 0].set_xlabel('Position (Mb)', fontsize=10)            axes[i, 1].set_xlabel('Position (Mb)', fontsize=10)        plt.suptitle('Colocalization: AD GWAS × Colon eQTL (Full Summary Statistics)',                  fontsize=14, fontweight='bold', y=1.01)    plt.tight_layout()    plt.savefig(f'{FIG}/Coloc_locuszoom_full_eQTL.png', dpi=300, bbox_inches='tight')    plt.savefig(f'{FIG}/Coloc_locuszoom_full_eQTL.pdf', dpi=300, bbox_inches='tight')    print(f"Saved LocusZoom figure ({n_plots} loci)")

## Step 7: Summary Figure — PP.H4 Bar Chart

In [ ]:
# PP.H4 bar chart for all genes × tissuesfig, ax = plt.subplots(figsize=(10, 6))genes_order = ['ADAM10', 'PICALM', 'CR1', 'CD2AP', 'APP', 'SORL1', 'PSEN1', 'TREM2', 'SPI1', 'BIN1']colors_t = {'Colon_Transverse': '#E74C3C', 'Colon_Sigmoid': '#3498DB'}x = np.arange(len(genes_order))width = 0.35for i, tissue in enumerate(tissues.keys()):    vals = []    for gene in genes_order:        sub = summary[(summary['gene']==gene) & (summary['tissue']==tissue)]        vals.append(sub['PP_H4'].values[0] if len(sub) > 0 else 0)    ax.bar(x + i*width - width/2, vals, width,            color=colors_t[tissue], alpha=0.85, label=tissue.replace('_',' '), edgecolor='white')ax.axhline(0.8, color='green', ls='--', alpha=0.5, label='Strong coloc (0.8)')ax.axhline(0.5, color='orange', ls='--', alpha=0.5, label='Moderate coloc (0.5)')ax.set_xticks(x)ax.set_xticklabels(genes_order, fontsize=11, fontweight='bold')ax.set_ylabel('PP.H4 (shared causal variant)', fontsize=12)ax.set_title('Colocalization: AD GWAS × Colon eQTL\n(Full GTEx V8 Summary Statistics)', fontsize=13, fontweight='bold')ax.legend(fontsize=9)ax.set_ylim(0, 1.05)# Add separator between APP and myeloid genesax.axvline(6.5, color='black', ls=':', alpha=0.3)ax.text(3, 1.0, 'APP pathway', ha='center', fontsize=10, fontstyle='italic', color='#E74C3C')ax.text(8, 1.0, 'Myeloid', ha='center', fontsize=10, fontstyle='italic', color='#3498DB')plt.tight_layout()plt.savefig(f'{FIG}/Coloc_PP_H4_summary_full_eQTL.png', dpi=300, bbox_inches='tight')plt.savefig(f'{FIG}/Coloc_PP_H4_summary_full_eQTL.pdf', dpi=300, bbox_inches='tight')print("Summary figure saved")

## Step 8: Comparison — Previous (Significant Only) vs Current (Full Data)

In [ ]:
# Compare with previous results using only significant eQTLsprev_results = {    ('PICALM', 'Colon_Transverse'): 0.1525,    ('PICALM', 'Colon_Sigmoid'): 0.8789,    ('APP', 'Colon_Transverse'): 0.1993,    ('APP', 'Colon_Sigmoid'): 0.2006,    ('ADAM10', 'Colon_Transverse'): 0.8661,    ('ADAM10', 'Colon_Sigmoid'): 0.8357,    ('CR1', 'Colon_Sigmoid'): 0.9923,}print("Comparison: Significant-only vs Full summary statistics")print(f"{'Gene':10s} {'Tissue':20s} {'Prev PP.H4':>11s} {'Full PP.H4':>11s} {'Change'}")print("-" * 70)for _, r in summary.iterrows():    key = (r['gene'], r['tissue'])    prev = prev_results.get(key, None)    if prev is not None:        change = r['PP_H4'] - prev        arrow = '↑' if change > 0.05 else '↓' if change < -0.05 else '~'        print(f"{r['gene']:10s} {r['tissue']:20s} {prev:11.4f} {r['PP_H4']:11.4f} {arrow} {change:+.4f}")

## ConclusionsThe full summary statistics colocalization analysis provides the methodologically correct assessment of whether AD GWAS and colon eQTL signals share causal variants.Key changes from the significant-only analysis:- N_SNPs per locus increases from ~40-200 to ~5,000-15,000- PP.H4 estimates are now properly calibrated against the full background of non-associated SNPs